# Nomobug Sales, Payments, Warranty and Refund Data

Beginner-friendly CP2 notebook. Run the cells from top to bottom.

It loads **all populated rows** but only displays small previews so the notebook stays responsive. Credentials and real source IDs stay in the ignored local `secrets/` and `data/profiles/` folders.

## 1. Import pandas and connect to the existing local Google OAuth files

In [ ]:
from pathlib import Path
import json
import re
import pandas as pd
from IPython.display import display
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

token_path = project_root / "secrets" / "google_token.json"
metadata_path = project_root / "data" / "profiles" / "google_source_metadata.json"

print("Project folder:", project_root)
print("Token found:", token_path.exists())
print("Source metadata found:", metadata_path.exists())

In [ ]:
scopes = [
    "https://www.googleapis.com/auth/spreadsheets.readonly",
    "https://www.googleapis.com/auth/calendar.readonly",
]

credentials = Credentials.from_authorized_user_file(str(token_path), scopes)
sheets_service = build("sheets", "v4", credentials=credentials, cache_discovery=False)
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))

print("Google Sheets connection ready")

## 2. Find the Session and Payment workbook

In [ ]:
workbook_title = "SESSION & PAYMENT (NMB)- CHIA"
spreadsheet_id = ""

for workbook in metadata["spreadsheets"]:
    if workbook["properties"]["title"] == workbook_title:
        spreadsheet_id = workbook["spreadsheetId"]

print("Workbook found:", bool(spreadsheet_id))

## 3. Select the useful tabs

`LTV` is excluded. `Commercial Clients` is loaded only for validation because it is auto-filled from SALES.

In [ ]:
tab_header_rows = {
    "SALES": 4,
    "PAYMENTS": 1,
    "PAYMENT LINK": 1,
    "WARRANTY CLAIM": 1,
    "Commercial Clients": 1,
    "REFUND": 1,
    "Recurring Payments ⭐": 1,
}

print(list(tab_header_rows))

## 4. Load every populated row into pandas DataFrames

Each DataFrame is stored inside the `tables` dictionary using its tab name.

In [ ]:
tables = {}
summary_rows = []

for tab_name, header_row in tab_header_rows.items():
    response = sheets_service.spreadsheets().values().get(
        spreadsheetId=spreadsheet_id,
        range=f"'{tab_name}'",
        majorDimension="ROWS",
    ).execute()

    rows = response.get("values", [])
    width = max(len(row) for row in rows)
    original_columns = rows[header_row - 1] + [""] * (width - len(rows[header_row - 1]))
    original_columns = [name.strip() or f"unnamed_{i + 1}" for i, name in enumerate(original_columns)]
    original_columns = [f"unnamed_{i + 1}" if "@" in name else name for i, name in enumerate(original_columns)]

    counts = {}
    columns = []
    for name in original_columns:
        counts[name] = counts.get(name, 0) + 1
        columns.append(name if counts[name] == 1 else f"{name}_{counts[name]}")

    data = [(row + [""] * width)[:width] for row in rows[header_row:]]
    df = pd.DataFrame(data, columns=columns)
    df = df.replace(r"^\s*$", pd.NA, regex=True).dropna(how="all")
    tables[tab_name] = df

    summary_rows.append({
        "tab": tab_name,
        "rows": len(df),
        "columns": len(df.columns),
        "exact_duplicates": int(df.duplicated().sum()),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

## 5. Give the main DataFrames simple names and preview them

In [ ]:
sales_df = tables["SALES"]
payments_df = tables["PAYMENTS"]
payment_links_df = tables["PAYMENT LINK"]
warranty_df = tables["WARRANTY CLAIM"]
commercial_df = tables["Commercial Clients"]
refund_df = tables["REFUND"]
recurring_df = tables["Recurring Payments ⭐"]

print("SALES preview")
display(sales_df.head())

print("PAYMENTS preview")
display(payments_df.head())

print("WARRANTY preview")
display(warranty_df.head())

## Important business rules

- SALES is used for closing details, package, price, deposit and balance.
- SALES service-date columns are not reliable; Calendar is the accurate source.
- PAYMENTS records money received and links using Customer ID.
- PAYMENT LINK uses phone because it has no Customer ID.
- Commercial Clients must not be added to SALES totals.
- Refund bank details must never enter analytical outputs.
- Recurring Payments is derived from SALES and must not be counted as another sale.

## 6. Create simple non-sensitive profiles

In [ ]:
quality_rows = []

for tab_name, df in tables.items():
    for column in df.columns:
        quality_rows.append({
            "tab": tab_name,
            "column": column,
            "non_null_count": int(df[column].notna().sum()),
            "missing_percent": round(float(df[column].isna().mean() * 100), 2),
            "distinct_count": int(df[column].nunique(dropna=True)),
        })

quality_df = pd.DataFrame(quality_rows)
display(quality_df)

## 7. Save all rows locally

Bank account numbers remain only in the protected raw REFUND file and must be dropped from future cleaned tables.

In [ ]:
raw_folder = project_root / "data" / "raw" / "session_payment"
profile_folder = project_root / "data" / "profiles" / "session_payment"
raw_folder.mkdir(parents=True, exist_ok=True)
profile_folder.mkdir(parents=True, exist_ok=True)

for tab_name, df in tables.items():
    safe_name = re.sub(r"[^A-Za-z0-9_-]+", "_", tab_name).strip("_")
    df.to_csv(raw_folder / f"{safe_name}.csv", index=False)

summary_df.to_csv(profile_folder / "session_payment_workbook_summary.csv", index=False)
quality_df.to_csv(profile_folder / "session_payment_column_profile.csv", index=False)

print("Saved all rows to:", raw_folder)
print("Saved profiles to:", profile_folder)